# Mirai Studio — Anime GPU on a free Colab runtime

This notebook runs **Automatic1111** (the image backend Mirai Studio talks to) and prints a **public URL** at the end.

**Before you start — do this first:**
1. Top menu → **Runtime → Change runtime type → Hardware accelerator: GPU (T4)** → Save.
2. Then run the cells in order, top to bottom.

> Why A1111 and not ComfyUI: A1111 has `--share` built in, which hands you a
 `*.gradio.live` URL with no tunnel to install. ComfyUI needs an extra
 `cloudflared` step. If you specifically want ComfyUI (video workflows), see
 the last cell.

In [ ]:
# 1) Confirm a real GPU is attached (must say Tesla T4 or similar).
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# 2) Install Automatic1111 + the API server.
#    --no-download-sd-model skips the default photorealistic checkpoint
#    (we do NOT want it — we want an anime one).
import os
if not os.path.isdir('stable-diffusion-webui'):
    !git clone https://github.com/AUTOMATIC1111/stable-diffusion-webui
%cd stable-diffusion-webui
!pip install -r requirements.txt
%cd ..

In [ ]:
# 3) Load an ANIME checkpoint.
#    Paste a direct .safetensors URL below, then run this cell.
#    Civitai gives one on each model page; Hugging Face mirrors do too.
MODEL_URL = ""   # a DIRECT .safetensors URL (no login page)

if not MODEL_URL:
    print('Set MODEL_URL to a direct .safetensors URL, then re-run this cell.')
else:
    !curl -L --fail -o "stable-diffusion-webui/models/Stable-diffusion/anime.safetensors" "{MODEL_URL}"
    !ls -la stable-diffusion-webui/models/Stable-diffusion/

In [ ]:
# 4) Start the API server with a PUBLIC URL (this runs until Colab stops).
#    Keep this cell running — it prints the gradio.live link Mirai needs.
#    --api enables the HTTP API the app calls.
#    --gradio-auth user:pass adds a password (optional but recommended).
%cd stable-diffusion-webui
!python launch.py --api --share --no-download-sd-model --xformers --gradio-auth mirai:mirai

## 5) Connect Mirai Studio

When cell 4 prints `Running on public URL: https://xxxxxxxx.gradio.live`:

1. Mirai Studio → **AI Marketplace** → **"Your Own GPU (Automatic1111)"** → *Use provider*.
2. **Settings → Providers → image → "Your GPU"**: paste that `gradio.live` URL.
3. Tick **"allow a remote GPU"** (the warning explains why — the URL is a secret).
4. Pick your anime checkpoint from the dropdown (the app reads what you installed).
5. Storyboard → **Generate (AI)**. 

> **Shut it down when done.** The public URL has no authentication beyond the
 `--gradio-auth` password. Anyone who learns it can use that GPU.

## Optional: ComfyUI instead (for video workflows)

ComfyUI has no `--share`. To reach it remotely, run it and add a tunnel:
```
!git clone https://github.com/comfyanonymous/ComfyUI && cd ComfyUI && pip install -r requirements.txt
!python main.py --listen &
!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared
!./cloudflared tunnel --url http://localhost:8188
```
Then paste the `trycloudflare.com` URL into **Settings → Providers → video → "Your GPU"** and paste your
workflow JSON there too.